# dots.mocr — Test Notebook (A100)
> **Runtime:** `Runtime > Change runtime type > A100 GPU`

In [ ]:
# 1. Install
!pip install -q "transformers==4.57.6" qwen_vl_utils accelerate
!pip install -q flash-attn --no-build-isolation

import os
if not os.path.exists("dots.mocr"):
    !git clone https://github.com/rednote-hilab/dots.mocr.git
os.chdir("/content/dots.mocr")
!pip install -e . -q
print("Done!")

In [ ]:
# 2. Load model
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "rednote-hilab/dots.mocr",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map="auto",
)
processor = AutoProcessor.from_pretrained("rednote-hilab/dots.mocr")
model.eval()
print(f"Loaded! VRAM: {torch.cuda.memory_allocated(0)/1e9:.1f} GB")

In [ ]:
# 3. Upload ảnh và chạy OCR
from google.colab import files as colab_files
from dots_mocr.utils import dict_promptmode_to_prompt
from PIL import Image
from io import BytesIO
import time

uploaded = colab_files.upload()

for filename, content in uploaded.items():
    img = Image.open(BytesIO(content)).convert("RGB")
    print(f"\nFile: {filename} | Size: {img.size}")

    # --- chọn mode tại đây ---
    MODE = "prompt_layout_all_en"
    # -------------------------

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text": dict_promptmode_to_prompt[MODE]},
        ],
    }]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to(model.device)

    t0 = time.time()
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=8192)
    elapsed = time.time() - t0

    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    result = processor.batch_decode(trimmed, skip_special_tokens=True)[0]

    print(f"Mode: {MODE} | Time: {elapsed:.1f}s")
    print("\n" + "="*60)
    print(result)

In [ ]:
# 4. Xem tất cả prompt modes có sẵn
for i, mode in enumerate(dict_promptmode_to_prompt, 1):
    print(f"{i}. {mode}")